In [74]:
import pandas as pd

df = pd.read_excel('C:/Users/egor2/trainee_de/task2_data/task_2_data_ex.xlsx')
print("rows:", len(df))
print(df)

df_agg = df.groupby([
    'plant_id', 'year', 'produced_material', 'component_material',
    'produced_material_release_type', 'produced_material_production_type',
    'component_material_release_type', 'component_material_production_type'
]).agg({
    'produced_material_quantity': 'sum',
    'component_material_quantity': 'sum'
}).reset_index()
print("aggreg rows:", len(df_agg))

print(df_agg)

all_materials = set(df_agg['produced_material'])
all_components = set(df_agg['component_material'])
fin_materials = all_materials - all_components
print( fin_materials)

fin_material_combinations = df_agg[df_agg['produced_material'].isin(fin_materials)][['plant_id', 'year', 'produced_material']].drop_duplicates()
print(fin_material_combinations)

release_type_dict = df_agg[['produced_material', 'produced_material_release_type']].drop_duplicates('produced_material').set_index('produced_material')['produced_material_release_type'].to_dict()
prod_type_dict = df_agg[['produced_material', 'produced_material_production_type']].drop_duplicates('produced_material').set_index('produced_material')['produced_material_production_type'].to_dict()

components_dict = df_agg.groupby('produced_material')['component_material'].apply(list).to_dict()
print("Components dict:", components_dict)

def build_hierarchy(fin_material, current_material, result_list, plant, year):
    components = components_dict.get(current_material, [])
    for component in components:
        if current_material != fin_material:
            row = {
                'plant': plant,
                'year': year,
                'fin_material': fin_material,
                'release_type_fin': release_type_dict.get(fin_material, 'Unknown'),
                'prod_type_fin': prod_type_dict.get(fin_material, None),
                'material': current_material,
                'release_type': release_type_dict.get(current_material, 'Unknown'),
                'prod_type': prod_type_dict.get(current_material, None),
                'component': component
            }
            result_list.append(row)
        build_hierarchy(fin_material, component, result_list, plant, year)

result_list = []
for _, row in fin_material_combinations.iterrows():
    plant = row['plant_id']
    year = row['year']
    fin_material = row['produced_material']
    build_hierarchy(fin_material, fin_material, result_list, plant, year)

result_df = pd.DataFrame(result_list)
print("Total rows:", len(result_df))
print(result_df)

print("2025:", len(result_df[result_df['year'] == 2025]))
print("PLANT_15:", len(result_df[result_df['plant'] == 'PLANT_15']))

result_df.to_excel('bom_output.xlsx', index=False)


rows: 1327
      year  month  produced_material  produced_material_production_type  \
0     2024      1              10000                               8002   
1     2024      1              50000                               8002   
2     2024      1              50000                               8002   
3     2024      1              50000                               8002   
4     2024      1              80070                               8007   
...    ...    ...                ...                                ...   
1322  2000      5                802                                 82   
1323  2000      5                802                                 82   
1324  2000      5                803                                 83   
1325  2000      5                804                                 84   
1326  2000      5                805                                 85   

     produced_material_release_type  produced_material_quantity  \
0                    

PermissionError: [Errno 13] Permission denied: 'bom_output.xlsx'

In [60]:
import pandas as pd

keys = ['a','b','c','d','c','b','a']
values = [1, 2, 3, 4, 33, 2, 11]

df = pd.DataFrame({'key': keys, 'value': values})

#df = df.drop_duplicates('key')

mdict = df.set_index('key')['value'].to_dict()

lookup_keys = ['a', 'c']
result = [mdict[k] for k in lookup_keys]

print("Dictionary:", mdict)
print("Lookup result:", result)


Dictionary: {'a': 1, 'b': 2, 'c': 3, 'd': 4}
Lookup result: [1, 3]
